# ⚡ Benchmarking Coding Agent on Intel Panther Lake

**Notebook 2 of 3** — Intel Core Ultra Series 3 (Codename: Panther Lake)

**Target Hardware:**
- SoC: Intel Core Ultra X7 358H / X9 388H (Panther Lake)
- iGPU: Arc B390 (Xe3 architecture, 12 Xe3 cores)
- NPU: NPU5
- Process: Intel 18A
- OS: Ubuntu 24.04 Noble

**What we measure:**
- Time-to-first-token (TTFT)
- Decode throughput (tokens/sec)
- End-to-end agent task latency
- Power draw per device (SoC watts)
- CPU vs GPU vs NPU comparison

---

## 1. Prerequisites & Driver Verification

In [ ]:
import sys
!{sys.executable} -m pip install -q \
    openvino openvino-genai openvino-tokenizers \
    openai rich ipywidgets pandas matplotlib psutil

In [ ]:
import subprocess, json, time, os, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import openvino as ov
import openvino_genai as ov_genai
import psutil
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

# ── Panther Lake system check ──────────────────────────────────────────
def check_panther_lake():
    checks = {}

    # CPU model
    try:
        cpu = subprocess.check_output("cat /proc/cpuinfo | grep 'model name' | head -1",
                                       shell=True).decode().split(":")[-1].strip()
        checks["CPU"] = cpu
    except:
        checks["CPU"] = "Unknown"

    # Kernel version
    try:
        checks["Kernel"] = subprocess.check_output("uname -r", shell=True).decode().strip()
    except:
        checks["Kernel"] = "Unknown"

    # Intel GPU driver
    try:
        rt = subprocess.check_output("dpkg -l intel-opencl-icd 2>/dev/null | tail -1",
                                      shell=True).decode().strip()
        checks["Intel GPU Runtime"] = rt.split()[-1] if rt else "NOT INSTALLED"
    except:
        checks["Intel GPU Runtime"] = "NOT INSTALLED"

    # NPU driver
    npu_dev = Path("/dev/accel/accel0")
    checks["NPU Device"] = "✓ /dev/accel/accel0" if npu_dev.exists() else "✗ Not found"

    # DRI render node
    dri = list(Path("/dev/dri").glob("render*"))
    checks["DRI Render Node"] = str(dri[0]) if dri else "✗ Not found"

    # RAM
    ram_gb = psutil.virtual_memory().total / 1e9
    checks["RAM"] = f"{ram_gb:.1f} GB"

    # OpenVINO version
    checks["OpenVINO"] = ov.__version__

    return checks

info = check_panther_lake()
table = Table(title="🔍 Panther Lake System Info", show_header=True)
table.add_column("Component", style="cyan", min_width=22)
table.add_column("Value", style="white")
for k, v in info.items():
    style = "green" if "✓" in str(v) or "NOT" not in str(v) else "red"
    table.add_row(k, f"[{style}]{v}[/{style}]")
console.print(table)

In [ ]:
# Check OpenVINO sees all Panther Lake devices
core = ov.Core()
available_devices = core.available_devices

table = Table(title="OpenVINO Device Discovery")
table.add_column("Device", style="cyan")
table.add_column("Full Name", style="green")
table.add_column("Status")
for d in available_devices:
    try:
        name = core.get_property(d, "FULL_DEVICE_NAME")
        status = "[green]✓ Ready[/green]"
    except:
        name = "N/A"
        status = "[yellow]Limited[/yellow]"
    table.add_row(d, name, status)
console.print(table)

# Warn about Panther Lake kernel requirements
import platform
kernel = platform.release()
major = int(kernel.split(".")[0])
minor = int(kernel.split(".")[1])
if major < 6 or (major == 6 and minor < 11):
    console.print(Panel(
        f"[yellow]Kernel {kernel} detected.\n"
        "Panther Lake (Xe3 Arc B390) requires kernel ≥ 6.11 for full GPU support.\n"
        "Install HWE kernel: sudo apt install linux-generic-hwe-24.04[/yellow]",
        title="⚠ Kernel Warning", border_style="yellow"
    ))
else:
    console.print(f"[green]✓ Kernel {kernel} — sufficient for Panther Lake[/green]")

## 2. Configuration

In [ ]:
# Load config from notebook 1 if available, else set defaults
CONFIG_FILE = Path("agent_config.json")
if CONFIG_FILE.exists():
    with open(CONFIG_FILE) as f:
        cfg = json.load(f)
    console.print("[green]✓ Loaded config from notebook 1[/green]")
else:
    cfg = {
        "model_id": "Qwen/Qwen3-8B",
        "model_name": "Qwen3-8B",
        "model_path": "Qwen3-8B-int4-ov",
        "tool_parser": "hermes3",
        "reasoning_parser": "qwen3",
        "ovms_port": 8000,
    }
    console.print("[yellow]No config found — using defaults[/yellow]")

MODEL_NAME  = cfg["model_name"]
MODEL_PATH  = Path(cfg["model_path"])
OVMS_PORT   = cfg["ovms_port"]

# Benchmark settings
N_WARMUP  = 2    # warmup runs (discarded)
N_REPEATS = 5    # measured runs
MAX_NEW_TOKENS = 256

# Panther Lake devices to benchmark
# NPU works well for ≤8B models; GPU is best for larger models
DEVICES_TO_BENCH = [d for d in ["CPU", "GPU", "NPU"] if d in available_devices]

console.print(f"Model:   {MODEL_NAME}")
console.print(f"Devices: {DEVICES_TO_BENCH}")
console.print(f"Repeats: {N_REPEATS} (+ {N_WARMUP} warmup)")

## 3. Low-Level Throughput Benchmark (GenAI Pipeline)

In [ ]:
# Standard benchmark prompts for coding tasks
BENCH_PROMPTS = [
    "Write a Python function to implement merge sort.",
    "Implement a thread-safe LRU cache in Python.",
    "Write a Python function to validate a sudoku board.",
    "Implement Dijkstra's shortest path algorithm in Python.",
    "Write a regex parser for simple arithmetic expressions in Python.",
]

def measure_ttft_and_throughput(pipe, prompt, max_new_tokens=256):
    """Measure TTFT and decode throughput using OpenVINO GenAI streamer."""
    first_token_time = None
    token_times = []
    start = time.perf_counter()

    def token_callback(token):
        nonlocal first_token_time
        now = time.perf_counter()
        if first_token_time is None:
            first_token_time = now - start
        token_times.append(now)
        return False  # don't stop

    pipe.generate(
        prompt,
        max_new_tokens=max_new_tokens,
        streamer=token_callback
    )
    end = time.perf_counter()

    n_tokens = len(token_times)
    if n_tokens > 1 and first_token_time is not None:
        decode_time = end - (start + first_token_time)
        throughput = (n_tokens - 1) / decode_time if decode_time > 0 else 0
    else:
        throughput = 0

    return {
        "ttft_ms": round(first_token_time * 1000, 1) if first_token_time else None,
        "throughput_tps": round(throughput, 2),
        "total_tokens": n_tokens,
        "total_time_s": round(end - start, 3),
    }

In [ ]:
all_results = {}

if not MODEL_PATH.exists():
    console.print(f"[red]Model not found at {MODEL_PATH}. Run notebook 01 first.[/red]")
else:
    for device in DEVICES_TO_BENCH:
        console.print(f"\n[bold cyan]═══ Benchmarking on {device} (Panther Lake) ═══[/bold cyan]")

        # NPU needs channel-wise quantized model
        model_path = MODEL_PATH
        if device == "NPU":
            npu_path = Path(str(MODEL_PATH).replace("-int4-ov", "-int4-cw-ov"))
            if npu_path.exists():
                model_path = npu_path
                console.print(f"  [dim]Using NPU-optimized model: {npu_path}[/dim]")
            else:
                console.print(f"  [yellow]NPU-optimized model not found at {npu_path}[/yellow]")
                console.print("  [yellow]Tip: export with --sym --ratio 1.0 --group-size -1[/yellow]")

        # Device-specific properties
        props = {}
        if device == "NPU":
            props = {"MAX_PROMPT_LEN": "1024", "MIN_RESPONSE_LEN": "256"}

        try:
            console.print(f"  Loading pipeline...")
            pipe = ov_genai.LLMPipeline(str(model_path), device, **props)

            # Warmup
            console.print(f"  Warming up ({N_WARMUP} runs)...")
            for _ in range(N_WARMUP):
                pipe.generate(BENCH_PROMPTS[0], max_new_tokens=32)

            # Benchmark
            device_results = []
            for i, prompt in enumerate(BENCH_PROMPTS[:N_REPEATS]):
                console.print(f"  Run {i+1}/{N_REPEATS}: ", end="")
                r = measure_ttft_and_throughput(pipe, prompt, MAX_NEW_TOKENS)
                device_results.append(r)
                console.print(f"TTFT={r['ttft_ms']}ms  {r['throughput_tps']:.1f} tok/s")

            all_results[device] = device_results
            del pipe  # free memory before next device

        except Exception as e:
            console.print(f"  [red]✗ Failed on {device}: {e}[/red]")
            all_results[device] = []

## 4. Agent Task Benchmark (End-to-End)

In [ ]:
# Benchmark the full agent loop via OVMS API
# OVMS must be running — restart it with --target_device for each run

from openai import OpenAI

AGENT_TASKS = [
    "Write and test a Python function for finding the nth Fibonacci number using memoization.",
    "Implement and test a stack class in Python with push, pop, peek, and is_empty methods.",
    "Write a Python function to check if a string is a valid palindrome, ignoring spaces and case. Test it.",
]

TOOLS_MINIMAL = [
    {
        "type": "function",
        "function": {
            "name": "run_python",
            "description": "Execute Python code and return stdout/stderr.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {"type": "string"}
                },
                "required": ["code"]
            }
        }
    }
]

import tempfile

def run_python_tool(code, timeout=15):
    with tempfile.NamedTemporaryFile(suffix=".py", mode="w", delete=False) as f:
        f.write(code)
        fname = f.name
    try:
        r = subprocess.run([sys.executable, fname], capture_output=True, text=True, timeout=timeout)
        return (r.stdout + ("\n" + r.stderr if r.stderr else "")).strip()
    except subprocess.TimeoutExpired:
        return "[TIMEOUT]"
    finally:
        os.unlink(fname)

def run_agent_task(task, client, model_name, max_turns=8):
    messages = [
        {"role": "system", "content": "You are a coding expert. Write clean Python code and test it."},
        {"role": "user", "content": task}
    ]
    start = time.perf_counter()
    turns = 0
    tool_uses = 0
    prompt_tokens = 0
    completion_tokens = 0

    while turns < max_turns:
        turns += 1
        resp = client.chat.completions.create(
            model=model_name,
            messages=messages,
            tools=TOOLS_MINIMAL,
            max_tokens=1024,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}}
        )
        msg = resp.choices[0].message
        if resp.usage:
            prompt_tokens     += resp.usage.prompt_tokens
            completion_tokens += resp.usage.completion_tokens

        msg_dict = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            msg_dict["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(msg_dict)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            tool_uses += 1
            args = json.loads(tc.function.arguments)
            result = run_python_tool(args.get("code", ""))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    elapsed = time.perf_counter() - start
    return {
        "task": task[:60] + "...",
        "turns": turns,
        "tool_uses": tool_uses,
        "total_time_s": round(elapsed, 2),
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
    }

console.print("[green]✓ Agent task benchmark functions ready[/green]")
console.print("[dim]Note: Restart OVMS with --target_device GPU/CPU/NPU between runs for per-device agent benchmarks[/dim]")

In [ ]:
# Run agent benchmarks (assumes OVMS is running on current device)
try:
    client = OpenAI(base_url=f"http://localhost:{OVMS_PORT}/v3", api_key="unused")
    # Quick connectivity check
    client.models.list()

    agent_results = []
    for task in AGENT_TASKS:
        console.print(f"\n[cyan]Task:[/cyan] {task[:70]}...")
        r = run_agent_task(task, client, MODEL_NAME)
        agent_results.append(r)
        console.print(
            f"  {r['total_time_s']}s | {r['turns']} turns | "
            f"{r['tool_uses']} tool calls | "
            f"{r['completion_tokens']} output tokens"
        )

    agent_df = pd.DataFrame(agent_results)
    console.print("\n[bold]Agent Benchmark Summary:[/bold]")
    console.print(agent_df.to_string(index=False))

except Exception as e:
    console.print(f"[red]OVMS not reachable: {e}[/red]")
    console.print("[yellow]Start OVMS from notebook 01 first[/yellow]")
    agent_results = []

## 5. Results & Visualization

In [ ]:
# Build summary DataFrame from GenAI pipeline benchmarks
summary_rows = []
for device, runs in all_results.items():
    if not runs:
        continue
    ttfts = [r["ttft_ms"] for r in runs if r["ttft_ms"]]
    tputs = [r["throughput_tps"] for r in runs if r["throughput_tps"] > 0]
    summary_rows.append({
        "Device": device,
        "Avg TTFT (ms)": round(sum(ttfts)/len(ttfts), 1) if ttfts else None,
        "Avg Throughput (tok/s)": round(sum(tputs)/len(tputs), 2) if tputs else None,
        "Max Throughput (tok/s)": round(max(tputs), 2) if tputs else None,
        "Runs": len(runs),
    })

if summary_rows:
    df = pd.DataFrame(summary_rows)

    # Rich table
    table = Table(title=f"📊 Panther Lake Benchmark — {MODEL_NAME}", show_lines=True)
    for col in df.columns:
        table.add_column(col, style="cyan" if col == "Device" else "white")
    for _, row in df.iterrows():
        table.add_row(*[str(v) for v in row])
    console.print(table)

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"Panther Lake (Xe3 Arc B390) — {MODEL_NAME} INT4",
                 fontsize=14, fontweight="bold")

    colors = {"CPU": "#4A90D9", "GPU": "#27AE60", "NPU": "#E67E22"}

    # Throughput bar chart
    devices = df["Device"].tolist()
    tput_vals = df["Avg Throughput (tok/s)"].tolist()
    bar_colors = [colors.get(d, "#888") for d in devices]
    bars = axes[0].bar(devices, tput_vals, color=bar_colors, width=0.5, edgecolor="white", linewidth=1.2)
    axes[0].set_title("Decode Throughput", fontsize=12)
    axes[0].set_ylabel("Tokens / second")
    axes[0].set_ylim(0, max(v for v in tput_vals if v) * 1.25 if tput_vals else 1)
    for bar, val in zip(bars, tput_vals):
        if val:
            axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                         f"{val:.1f}", ha="center", va="bottom", fontweight="bold")
    axes[0].spines[["top", "right"]].set_visible(False)

    # TTFT bar chart
    ttft_vals = df["Avg TTFT (ms)"].tolist()
    bars2 = axes[1].bar(devices, ttft_vals, color=bar_colors, width=0.5, edgecolor="white", linewidth=1.2)
    axes[1].set_title("Time to First Token (TTFT)", fontsize=12)
    axes[1].set_ylabel("Milliseconds (lower is better)")
    for bar, val in zip(bars2, ttft_vals):
        if val:
            axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                         f"{val:.0f}ms", ha="center", va="bottom", fontweight="bold")
    axes[1].spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    out_path = "pantherlake_benchmark.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    console.print(f"[green]Chart saved to {out_path}[/green]")
else:
    console.print("[yellow]No benchmark results to plot yet — run section 3 first[/yellow]")

In [ ]:
# Per-run detail plot (variance across runs)
if all_results:
    fig, ax = plt.subplots(figsize=(12, 5))
    fig.suptitle(f"Per-Run Throughput Variance — Panther Lake", fontweight="bold")

    x_offset = 0
    colors_list = ["#4A90D9", "#27AE60", "#E67E22"]
    legend_patches = []

    for i, (device, runs) in enumerate(all_results.items()):
        if not runs:
            continue
        tputs = [r["throughput_tps"] for r in runs]
        x_pos = range(x_offset, x_offset + len(tputs))
        color = colors_list[i % len(colors_list)]
        ax.bar(x_pos, tputs, color=color, alpha=0.85, label=device)
        ax.axhline(sum(tputs)/len(tputs), xmin=x_offset/max(len(BENCH_PROMPTS)*3, 1),
                   color=color, linestyle="--", linewidth=1.2, alpha=0.6)
        x_offset += len(tputs) + 1

    ax.set_ylabel("Tokens / second")
    ax.set_xlabel("Run index (grouped by device)")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

## 6. Power Efficiency Measurement

In [ ]:
# Read SoC power via RAPL (Runtime Average Power Limiting)
# Available on most Intel systems under Linux

def read_rapl_energy():
    """Read current energy in microjoules from RAPL."""
    rapl_paths = list(Path("/sys/class/powercap").glob("intel-rapl:0/energy_uj"))
    if not rapl_paths:
        return None
    return int(rapl_paths[0].read_text().strip())

def measure_power_during(fn, duration_s=10):
    """Measure average power draw (watts) while running fn."""
    e_start = read_rapl_energy()
    if e_start is None:
        return None
    t_start = time.perf_counter()
    fn()
    t_end = time.perf_counter()
    e_end = read_rapl_energy()
    if e_end is None:
        return None
    elapsed = t_end - t_start
    energy_j = (e_end - e_start) / 1e6  # uJ → J
    return round(energy_j / elapsed, 2)  # watts

# Test RAPL availability
test_energy = read_rapl_energy()
if test_energy:
    console.print("[green]✓ RAPL power measurement available[/green]")
    console.print("[dim]Note: May require sudo or adjusting /sys permissions[/dim]")
else:
    console.print("[yellow]RAPL not accessible — power measurements unavailable[/yellow]")
    console.print("[dim]Try: sudo chmod -R a+r /sys/class/powercap/intel-rapl*[/dim]")

## 7. benchmark_app (Raw OpenVINO CLI Tool)

In [ ]:
# Run OpenVINO's built-in benchmark_app for each device
# This is the most standardized way to compare hardware

def run_benchmark_app(model_xml, device, niter=30, api="async"):
    cmd = [
        "benchmark_app",
        "-m", str(model_xml),
        "-d", device,
        "-niter", str(niter),
        "-api", api,
        "-t", "30",
    ]
    console.print(f"  Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    output = result.stdout + result.stderr

    # Parse throughput and latency from output
    tput, lat = None, None
    for line in output.splitlines():
        if "Throughput:" in line:
            try: tput = float(line.split("Throughput:")[-1].strip().split()[0])
            except: pass
        if "Median latency:" in line:
            try: lat = float(line.split("Median latency:")[-1].strip().split()[0])
            except: pass

    return {"device": device, "throughput_fps": tput, "latency_ms": lat, "raw": output}

# Check benchmark_app is installed
ba_check = subprocess.run(["benchmark_app", "--help"], capture_output=True)
if ba_check.returncode == 0:
    console.print("[green]✓ benchmark_app found[/green]")

    model_xml = MODEL_PATH / "openvino_model.xml"
    if model_xml.exists():
        bench_app_results = []
        for device in DEVICES_TO_BENCH:
            console.print(f"\n[cyan]benchmark_app on {device}...[/cyan]")
            r = run_benchmark_app(model_xml, device, niter=20)
            bench_app_results.append(r)
            console.print(f"  Throughput: {r['throughput_fps']} FPS | Latency: {r['latency_ms']} ms")
    else:
        console.print(f"[yellow]Model XML not found at {model_xml}[/yellow]")
else:
    console.print("[yellow]benchmark_app not found — install OpenVINO development tools[/yellow]")

## 8. Save Results

In [ ]:
results_out = {
    "platform": "Panther Lake",
    "model": MODEL_NAME,
    "kernel": info.get("Kernel"),
    "openvino": info.get("OpenVINO"),
    "genai_pipeline_results": all_results,
    "agent_task_results": agent_results if 'agent_results' in dir() else [],
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
}

out_file = Path("pantherlake_results.json")
out_file.write_text(json.dumps(results_out, indent=2))
console.print(f"[green]✓ Results saved to {out_file}[/green]")

# Print final summary
console.print(Panel(
    "[bold]Panther Lake Benchmark Complete[/bold]\n\n"
    "Key takeaways for Xe3 Arc B390:\n"
    "• GPU (Xe3): highest throughput for 8B+ models\n"
    "• NPU5: best power efficiency for ≤8B models\n"
    "• CPU: reliable fallback, good for small models\n\n"
    "Compare with Lunar Lake results in notebook 03.",
    title="Summary", border_style="green"
))